# XBTorch::Example 04:Compare Device Models (Time)

## Introduction

In this example, we will develop a better understanding on the time complexity of the tabular device models against the analytical device models. We will simulate dummy neural networks at increasing depths (# of hidden layers), timing the overall conductance update operation of the network over a series of dummy epochs, where the device model update is simulated without ever carrying the update forward to the next epoch (in essence, this is the same as setting network learning rate to 0).

## Getting Started

Let's import necessary dependencies.

In [1]:
# General Imports
import numpy as np
import random
import time
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

In [2]:
# XBTorch Imports
import xbtorch.optim as xboptim
from xbtorch.patches import xbtorch_model
from xbtorch.devices import AnalyticalReal, TabularAnalyticalReal

import xbtorch

W0930 15:31:05.152000 1742600 torch/utils/cpp_extension.py:2425] TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
W0930 15:31:05.152000 1742600 torch/utils/cpp_extension.py:2425] If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'] to specific architectures.


## Defining the Network

Let's define our dummy neural network. This will be a simple multi-layer perceptron network where the number of hidden layers is a parameter.

In [3]:
class DummyMLP(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_hidden_layers):
        super(DummyMLP, self).__init__()
        
        layers = [nn.Linear(input_size, hidden_size, bias=False), nn.ReLU()]
        
        for _ in range(num_hidden_layers - 1):
            layers.append(nn.Linear(hidden_size, hidden_size, bias=False))
            layers.append(nn.ReLU())
        
        layers.append(nn.Linear(hidden_size, output_size, bias=False))
        
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

Let's also define our network hyperparameters. We will be sweeping the number of hidden layers for each device model. Everything else will remain constant.

In [4]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Sweep the hidden layer size
num_hidden_layerss = [1, 5, 10, 20]

batch_size = 4096
lr = 1
n_epochs = 500
input_size = 784
hidden_size = 500
output_size = 10

# Dummy inputs for the dummy network
dummy_inputs = torch.randn(batch_size, input_size).to(device) # inputs for our dummy network
class_indices = torch.randint(0, output_size, (batch_size,))
dummy_labels = torch.eye(output_size)[class_indices].to(device)

criterion = nn.CrossEntropyLoss()

analytical_device = AnalyticalReal()
tabular_device = TabularAnalyticalReal()


device_models = [analytical_device, tabular_device]
device_model_names = ["Analytical", "Tabular"]

Finally, let's perform the actual sweep.

In [ ]:
SEED = 512

time_dict = {}

for device_model in device_models:
    
    # fix seeds for each device model
    np.random.seed(SEED)
    random.seed(SEED)
    torch.manual_seed(SEED)
    
    # initialize time list for this device model
    time_dict[device_model] = {}

    # sweep over number of hidden layers
    for num_hidden_layers in num_hidden_layerss:

        # initialize xbtorch with the device model
        xbtorch.initialize(device_type=device_model)
        
        # initialize time list for this device model
        time_dict[device_model][num_hidden_layers] = np.zeros((n_epochs, 1))
        
        # create and patch the mlp
        mlp = DummyMLP(input_size, hidden_size, output_size, num_hidden_layers)
        mlp = xbtorch_model(mlp)
        mlp = mlp.to(device)

        optimizer = xboptim.SGD(mlp.parameters(), lr=lr)

        # repeat iters to simulate epochs (forward + backward)
        for epoch in range(n_epochs): # iterate multiple times
            start = time.time()

            dummy_outputs = mlp(dummy_inputs)
            loss = criterion(dummy_outputs, dummy_labels)
            
            optimizer.zero_grad()
            loss.backward()
            # optimizer.step() # we intentionally do not update network weights, the idea is to simply compute the time it takes to compute the updated weight inside the device model's weight

            end = time.time()

            time_dict[device_model][num_hidden_layers][epoch] = (end - start)

        # reset cached parameters for the device model
        device_model.reset_cached_params()

/tmp/ipykernel_1742600/2043148239.py:39: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  loss.backward()


In [ ]:
figs = []

for num_hidden_layers in num_hidden_layerss:
    plt.title(f"# of hidden layers: {num_hidden_layers}")
    for idx, device_model in enumerate(device_models):
        plt.plot(range(n_epochs-1), time_dict[device_model][num_hidden_layers][1:], label=device_model_names[idx]) # skip 1st to drop pytorch initialization
        print("Average Time", np.average(time_dict[device_model][num_hidden_layers][1:]), np.std(time_dict[device_model][num_hidden_layers][1:]))
        plt.xlabel("Epoch (#)")
        plt.ylabel("Time (s)")
    
    plt.legend()
    plt.show()

## Conclusion
We have seen how based on the network size the overall time complexity of one model could be better than the other. Although the two models utilized here are identical in terms of their device-level behavior, the tabular approach is faster in shallow networks and the analytical approach is faster with deeper networks.